# W03 Robot Performance Data Pipeline

**Prepared by:** Yincheng Zhang

## Objective

Build a reproducible Python pipeline that reads the KPI framework and
public financial proxy data, performs data cleaning and feature engineering,
and generates a synthetic robot-performance dataset for the Week 4 dashboard.

The pipeline covers Aido Rover, Sentinel Prime AI, and Fari.

## Data Sources and Boundaries

- The KPI framework contains metric definitions, public proxy references,
  proposed targets, and source limitations.
- The Week 2 Knightscope dataset provides public financial context.
  Financial ratios are not used to infer robot operating performance.
- Operational records are synthetic because InGen internal deployment data
  are unavailable.
- Simulation ranges are analyst-selected assumptions informed by the
  framework's proxy references and proposed targets. They are not fitted
  empirical distributions or estimates of actual InGen performance.
- Source references are retained from the framework. This notebook does
  not independently verify the underlying publications.

## Observation Level

The primary output contains one row per platform, synthetic deployment unit,
and month:

3 platforms × 5 deployment units × 12 months = 180 records.

For Fari, a deployment unit represents a user cohort rather than an individual
user. The simulated period is January–December 2025 and does not represent
historical InGen operations.

A supporting table retains the numerator, denominator, and calculated value
for each KPI, producing 900 measurement rows.

The monthly dataset supports dashboard prototyping. It does not replace the
daily or weekly monitoring specified in the KPI framework.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

from IPython.display import display


pipeline_start = perf_counter()

RANDOM_STATE = 42

FRAMEWORK_PATH = Path("W03_KPI_Framework.xlsx")
PUBLIC_DATA_PATH = Path("W02_Market_Data.xlsx")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


for file_path in [FRAMEWORK_PATH, PUBLIC_DATA_PATH]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path.resolve()}"
        )


pd.set_option("display.max_columns", None)

print("Input files found.")

Input files found.


In [2]:
framework = pd.read_excel(
    FRAMEWORK_PATH,
    sheet_name="KPI_Framework"
)


# Normalize column names.
framework.columns = (
    framework.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\W+", "_", regex=True)
    .str.strip("_")
)


# Remove leading and trailing whitespace from text values.
for column in framework.select_dtypes(
    include=["object", "string"]
).columns:
    framework[column] = framework[column].map(
        lambda value: value.strip()
        if isinstance(value, str) else value
    )


framework = framework.dropna(how="all").drop_duplicates()


required_columns = [
    "platform",
    "kpi_name",
    "definition",
    "numerator",
    "denominator",
    "formula",
    "public_proxy_benchmark",
    "target_range",
    "reporting_frequency",
    "source",
    "limitation"
]

missing_columns = set(required_columns) - set(framework.columns)

assert not missing_columns, (
    f"Missing framework columns: {missing_columns}"
)

assert not framework.duplicated(
    ["platform", "kpi_name"]
).any(), "Duplicate platform/KPI definitions."

assert framework[
    ["platform", "kpi_name", "source"]
].notna().all().all()

assert set(framework["platform"]) == {
    "Aido Rover", "Sentinel Prime AI", "Fari"
}

assert len(framework) == 15

assert framework.groupby("platform").size().eq(5).all()


display(
    framework[
        [
            "platform",
            "kpi_name",
            "formula",
            "public_proxy_benchmark",
            "target_range"
        ]
    ]
)

print("Framework validation passed: 3 platforms and 15 KPIs.")

,platform,kpi_name,formula,public_proxy_benchmark,target_range
0,Aido Rover,Patrol Coverage Rate,(Area Covered / Total Designated Patrol Area) ...,96.5%–98.3% area coverage reported in autonomo...,Healthy: ≥95%; Review trigger: <90%
1,Aido Rover,Incident Detection Rate,(Correctly Detected Incidents / Total Verified...,96.5% abnormal-event detection in a patrol-rob...,Internally derived target: Healthy ≥95%; monit...
2,Aido Rover,False Alarm Rate,(False-Positive Alerts / Total Robot-Generated...,0.0% false alarm rate reported for Tiny-YOLOv3...,Internally derived target: Healthy ≤1.5%; perf...
3,Aido Rover,Uptime,(Operational Available Time / Scheduled Operat...,Commercial autonomous-robot fleet operators pu...,Internally derived target: Healthy ≥99%; monit...
4,Aido Rover,Energy Consumption per Patrol Hour,Total Energy Consumed (kWh) / Active Patrol Hours,Approximately 0.13–0.17 kWh per operating hour...,Internally derived target: Healthy ≤0.18 kWh p...
5,Sentinel Prime AI,Incident Detection Rate,(Correctly Detected Incidents / Total Verified...,Published video-surveillance anomaly-detection...,Internally derived target: Healthy ≥95%; monit...
6,Sentinel Prime AI,False Positive Rate,TP / (TP + FN),1.9% false alarm rate reported by a real-world...,Internally derived target: Healthy ≤2%; monito...
7,Sentinel Prime AI,Mean Operator Response Time,Σ(Operator Acknowledgement Time − Alert Genera...,6-minute mean time to acknowledge (MTTA) repor...,Internally derived target: Healthy ≤10 min; mo...
8,Sentinel Prime AI,Coverage Area,(Effectively Monitored Area / Total Designated...,91.80%–95.98% average coverage reported across...,Internally derived target: Healthy ≥95%; monit...
9,Sentinel Prime AI,System Availability,MTBF / (MTBF + MTTR) × 100,99.9%–99.95% availability as a high-availabili...,Internally derived target: Healthy ≥99.9%; mon...


Framework validation passed: 3 platforms and 15 KPIs.


## Measurement Decisions

1. Sentinel's "False Positive Rate" is implemented according to its written
   definition: false alerts divided by total generated alerts. The framework's
   formula TP / (TP + FN) is inconsistent with that definition and measures
   recall. The implemented metric is named `false_alert_pct`. It is not the
   conventional false-positive rate FP / (FP + TN). The cited research
   benchmark may use a different denominator and is not assumed equivalent.

2. Fari interaction frequency is calculated as interaction events divided by
   active user-days, matching the framework's per-day target. This differs
   from dividing monthly sessions by distinct monthly active users.

3. Brief interaction events and completed structured sessions are modeled
   separately. Average session duration applies only to completed structured
   sessions. Goal success uses initiated goal-oriented sessions as its
   denominator, including unsuccessful and incomplete sessions.

4. Sentinel availability follows the framework's MTBF / (MTBF + MTTR)
   formula. These are assumed reliability parameters, not estimates from
   observed monthly failure logs.

## Cleaning and Missing-Value Policy

- Remove exact duplicate records.
- Reject conflicting records with the same measurement key.
- Reject invalid dates, unexpected nonnumeric inputs, and negative quantities.
- Preserve missing measurement inputs rather than replacing them with zero.
- Return an unavailable KPI when its denominator is zero or an input is missing.
- Preserve missing public financial disclosures without interpolation.
- Leave metrics that do not apply to a platform blank in the wide output.

No statistical imputation is applied to incident counts, time totals, or
exposure denominators because it could create unsupported performance claims.

In [3]:
public_raw = pd.read_excel(
    PUBLIC_DATA_PATH,
    sheet_name="KSCP_Financials"
)

public_financials = (
    public_raw
    .set_index("Period")
    .T
    .rename_axis("period")
    .reset_index()
)

public_financials.columns.name = None

public_financials = public_financials.drop_duplicates()


# Convert "Q1 2025" into a sortable quarter-end date.
period_parts = public_financials["period"].str.extract(
    r"^Q([1-4])\s+(\d{4})$"
)

assert period_parts.notna().all().all(), (
    "Unexpected quarterly period format."
)

public_financials["period_end"] = (
    pd.PeriodIndex(
        period_parts[1] + "Q" + period_parts[0],
        freq="Q"
    )
    .to_timestamp(how="end")
    .normalize()
)

assert not public_financials["period_end"].duplicated().any()


financial_columns = [
    "Revenue",
    "Cost of Revenue",
    "Gross Profit",
    "Gross Margin",
    "R&D",
    "Sales, General and Administrative",
    "Cash & Cash Equivalents",
    "Net Cash Burn"
]

for column in financial_columns:
    public_financials[column] = pd.to_numeric(
        public_financials[column],
        errors="raise"
    )

public_financials = (
    public_financials
    .sort_values("period_end")
    .reset_index(drop=True)
)


# Ensure a one-row shift really means one consecutive quarter.
quarter_index = pd.PeriodIndex(
    public_financials["period_end"],
    freq="Q"
)

assert np.all(np.diff(quarter_index.asi8) == 1), (
    "Quarterly observations are not consecutive."
)

assert public_financials["Revenue"].notna().all()
assert public_financials["Revenue"].gt(0).all()


public_financials["revenue_qoq_growth"] = (
    public_financials["Revenue"].pct_change(fill_method=None)
)

public_financials["revenue_4q_mean"] = (
    public_financials["Revenue"]
    .rolling(4, min_periods=4)
    .mean()
)

public_financials["calculated_gross_margin"] = (
    public_financials["Revenue"]
    - public_financials["Cost of Revenue"]
) / public_financials["Revenue"]


display(
    public_financials[
        [
            "period",
            "Revenue",
            "revenue_qoq_growth",
            "revenue_4q_mean",
            "calculated_gross_margin"
        ]
    ]
)

,period,Revenue,revenue_qoq_growth,revenue_4q_mean,calculated_gross_margin
0,Q1 2025,2.917,NaN,NaN,-0.229002
1,Q2 2025,2.749,-0.057593,NaN,-0.333940
2,Q3 2025,3.131,0.138960,NaN,-0.501437
3,Q4 2025,2.538,-0.189396,2.83375,-0.637904
4,Q1 2026,6.016,1.370370,3.60850,0.077294
5,Q2 2026,9.024,0.500000,5.17725,0.073582


In [4]:
specifications = [
    (
        "Aido Rover", "Patrol Coverage Rate",
        "patrol_coverage_pct", 100, "%",
        "Uniform coverage fraction: 0.88–0.995"
    ),
    (
        "Aido Rover", "Incident Detection Rate",
        "incident_detection_pct", 100, "%",
        "Binomial detection probability: Uniform(0.88, 0.99)"
    ),
    (
        "Aido Rover", "False Alarm Rate",
        "false_alert_pct", 100, "%",
        "False alerts ~ Poisson(1.5); all alerts = true + false alerts"
    ),
    (
        "Aido Rover", "Uptime",
        "uptime_pct", 100, "%",
        "Uniform available fraction: 0.94–0.999"
    ),
    (
        "Aido Rover", "Energy Consumption per Patrol Hour",
        "energy_kwh_per_hour", 1, "kWh/hour",
        "Uniform energy intensity: 0.12–0.28"
    ),
    (
        "Sentinel Prime AI", "Incident Detection Rate",
        "incident_detection_pct", 100, "%",
        "Binomial detection probability: Uniform(0.88, 0.99)"
    ),
    (
        "Sentinel Prime AI", "False Positive Rate",
        "false_alert_pct", 100, "%",
        "False alerts ~ Poisson(2); all alerts = true + false alerts"
    ),
    (
        "Sentinel Prime AI", "Mean Operator Response Time",
        "operator_response_min", 1, "minutes",
        "Uniform monthly mean: 4–18 minutes"
    ),
    (
        "Sentinel Prime AI", "Coverage Area",
        "monitoring_coverage_pct", 100, "%",
        "Uniform coverage fraction: 0.88–0.99"
    ),
    (
        "Sentinel Prime AI", "System Availability",
        "availability_pct", 100, "%",
        "MTBF Uniform(300,800) hours; MTTR Uniform(0.1,6) hours"
    ),
    (
        "Fari", "Average Session Duration",
        "session_duration_min", 1, "minutes",
        "Uniform mean completed structured-session duration: 4–24 minutes"
    ),
    (
        "Fari", "Interaction Frequency",
        "interactions_per_active_user_day", 1,
        "interactions/active user-day",
        "Poisson events: mean = active user-days × Uniform(8,42)"
    ),
    (
        "Fari", "Active-Day Engagement Rate",
        "active_day_engagement_pct", 100, "%",
        "Binomial active-day probability: Uniform(0.55,0.95)"
    ),
    (
        "Fari", "Average Response Time",
        "response_latency_sec", 1, "seconds",
        "Uniform monthly mean: 3–12 seconds"
    ),
    (
        "Fari", "Goal-Oriented Interaction Success Rate",
        "goal_success_pct", 100, "%",
        "95% completion probability; conditional success Uniform(0.65,0.95)"
    )
]


metadata = pd.DataFrame(
    specifications,
    columns=[
        "platform", "kpi_name", "metric",
        "scale", "unit", "simulation_assumption"
    ]
)

metadata = metadata.merge(
    framework,
    on=["platform", "kpi_name"],
    how="left",
    validate="one_to_one"
)

assert len(metadata) == 15
assert metadata["source"].notna().all()
assert not metadata.duplicated(["platform", "metric"]).any()


metadata["implemented_calculation"] = (
    "numerator / denominator * scale"
)

metadata["implementation_note"] = (
    "Monthly prototype calculated from synthetic inputs."
)

metadata.loc[
    (metadata["platform"] == "Sentinel Prime AI")
    & (metadata["metric"] == "false_alert_pct"),
    "implementation_note"
] = (
    "False alerts / total alerts × 100. "
    "Corrects the framework's inconsistent recall formula. "
    "Research benchmark denominator requires comparability review."
)

metadata.loc[
    metadata["metric"] == "interactions_per_active_user_day",
    "implementation_note"
] = (
    "Interaction events / active user-days. "
    "Daily exposure normalization matches the stated per-day target."
)


display(
    metadata[
        [
            "platform", "kpi_name", "metric", "unit",
            "simulation_assumption", "implementation_note"
        ]
    ]
)

,platform,kpi_name,metric,unit,simulation_assumption,implementation_note
0,Aido Rover,Patrol Coverage Rate,patrol_coverage_pct,%,Uniform coverage fraction: 0.88–0.995,Monthly prototype calculated from synthetic in...
1,Aido Rover,Incident Detection Rate,incident_detection_pct,%,"Binomial detection probability: Uniform(0.88, ...",Monthly prototype calculated from synthetic in...
2,Aido Rover,False Alarm Rate,false_alert_pct,%,False alerts ~ Poisson(1.5); all alerts = true...,Monthly prototype calculated from synthetic in...
3,Aido Rover,Uptime,uptime_pct,%,Uniform available fraction: 0.94–0.999,Monthly prototype calculated from synthetic in...
4,Aido Rover,Energy Consumption per Patrol Hour,energy_kwh_per_hour,kWh/hour,Uniform energy intensity: 0.12–0.28,Monthly prototype calculated from synthetic in...
5,Sentinel Prime AI,Incident Detection Rate,incident_detection_pct,%,"Binomial detection probability: Uniform(0.88, ...",Monthly prototype calculated from synthetic in...
6,Sentinel Prime AI,False Positive Rate,false_alert_pct,%,False alerts ~ Poisson(2); all alerts = true +...,False alerts / total alerts × 100. Corrects th...
7,Sentinel Prime AI,Mean Operator Response Time,operator_response_min,minutes,Uniform monthly mean: 4–18 minutes,Monthly prototype calculated from synthetic in...
8,Sentinel Prime AI,Coverage Area,monitoring_coverage_pct,%,Uniform coverage fraction: 0.88–0.99,Monthly prototype calculated from synthetic in...
9,Sentinel Prime AI,System Availability,availability_pct,%,"MTBF Uniform(300,800) hours; MTTR Uniform(0.1,...",Monthly prototype calculated from synthetic in...


## Simulation Methodology

The generator creates five deployment units per platform over twelve months.

- Uniform distributions provide illustrative continuous values and
  probabilities within documented ranges.
- Poisson distributions generate nonnegative event counts.
- Binomial distributions generate detected incidents, active user-days,
  completed sessions, and successful sessions from their eligible totals.

Additional assumptions:

- Each security deployment has Poisson(80) verified incidents per month.
- Each verified detected incident produces one true alert. All alerts are
  reviewed, and Sentinel alerts are assumed to be acknowledged.
- Aido Rover has 12 scheduled operating hours per day and one designated-area
  patrol per day. Active patrol hours equal 60%–90% of available hours.
- Each security deployment has a fixed area drawn from Uniform(2,20) hectares.
- Each Fari cohort has 10–30 users, all eligible throughout the month.
- Fari initiates an expected 0.6 structured sessions per active user-day.
- Brief interaction events are counted separately from structured sessions.
- Fari active days assume at least one qualifying meaningful interaction.
  All simulated interaction events are assumed to receive a valid response.

These assumptions intentionally produce varied observations for testing the
dashboard. No seasonal trend, improvement trend, or causal relationship is
imposed. Patterns in the resulting data are simulation outcomes, not evidence
of actual product performance.

The synthetic inputs are monthly totals and assumed reliability parameters.
They are not reconstructed GPS traces, incident logs, or individual
conversation records.

In [5]:
# Reset the generator so this cell produces the same results when rerun.
rng = np.random.default_rng(RANDOM_STATE)

months = pd.date_range(
    "2025-01-01",
    periods=12,
    freq="MS"
)

platform_ids = {
    "Aido Rover": "AR",
    "Sentinel Prime AI": "SP",
    "Fari": "FA"
}

lookup = metadata.set_index(["platform", "metric"])

rows = []


for platform, prefix in platform_ids.items():

    for unit_number in range(1, 6):

        unit_id = f"{prefix}-{unit_number:02d}"

        # Keep deployment characteristics stable across months.
        area = rng.uniform(2, 20)
        users = int(rng.integers(10, 31))

        for month in months:

            days = month.days_in_month

            def emit(metric, numerator, denominator):
                info = lookup.loc[(platform, metric)]

                rows.append({
                    "month": month.strftime("%Y-%m-%d"),
                    "platform": platform,
                    "unit_id": unit_id,
                    "metric": metric,
                    "numerator": numerator,
                    "denominator": denominator,
                    "scale": info["scale"],
                    "unit": info["unit"],
                    "data_origin": "SYNTHETIC"
                })

            # Shared security-event logic.
            if platform in ["Aido Rover", "Sentinel Prime AI"]:

                incidents = int(rng.poisson(80))

                detected = int(
                    rng.binomial(
                        incidents,
                        rng.uniform(0.88, 0.99)
                    )
                )

                false_alerts = int(
                    rng.poisson(
                        1.5 if platform == "Aido Rover" else 2
                    )
                )

                alerts = detected + false_alerts

                emit(
                    "incident_detection_pct",
                    detected,
                    incidents
                )

                emit(
                    "false_alert_pct",
                    false_alerts,
                    alerts
                )

            if platform == "Aido Rover":

                # Monthly sum of designated area across daily patrols.
                designated_area = area * days

                emit(
                    "patrol_coverage_pct",
                    designated_area * rng.uniform(0.88, 0.995),
                    designated_area
                )

                scheduled_hours = days * 12
                available_hours = (
                    scheduled_hours * rng.uniform(0.94, 0.999)
                )

                emit(
                    "uptime_pct",
                    available_hours,
                    scheduled_hours
                )

                active_hours = (
                    available_hours * rng.uniform(0.6, 0.9)
                )

                emit(
                    "energy_kwh_per_hour",
                    active_hours * rng.uniform(0.12, 0.28),
                    active_hours
                )

            elif platform == "Sentinel Prime AI":

                emit(
                    "operator_response_min",
                    alerts * rng.uniform(4, 18),
                    alerts
                )

                emit(
                    "monitoring_coverage_pct",
                    area * rng.uniform(0.88, 0.99),
                    area
                )

                mtbf = rng.uniform(300, 800)
                mttr = rng.uniform(0.1, 6)

                emit(
                    "availability_pct",
                    mtbf,
                    mtbf + mttr
                )

            else:

                eligible_days = users * days

                active_days = int(
                    rng.binomial(
                        eligible_days,
                        rng.uniform(0.55, 0.95)
                    )
                )

                interactions = int(
                    rng.poisson(
                        active_days * rng.uniform(8, 42)
                    )
                )

                initiated = int(
                    rng.poisson(active_days * 0.6)
                )

                completed = int(
                    rng.binomial(initiated, 0.95)
                )

                successful = int(
                    rng.binomial(
                        completed,
                        rng.uniform(0.65, 0.95)
                    )
                )

                emit(
                    "session_duration_min",
                    completed * rng.uniform(4, 24),
                    completed
                )

                emit(
                    "interactions_per_active_user_day",
                    interactions,
                    active_days
                )

                emit(
                    "active_day_engagement_pct",
                    active_days,
                    eligible_days
                )

                emit(
                    "response_latency_sec",
                    interactions * rng.uniform(3, 12),
                    interactions
                )

                emit(
                    "goal_success_pct",
                    successful,
                    initiated
                )


raw_measurements = pd.DataFrame(rows)

print(
    f"Generated {len(raw_measurements):,} "
    "synthetic KPI measurement records."
)

display(raw_measurements.head())

Generated 900 synthetic KPI measurement records.


,month,platform,unit_id,metric,numerator,denominator,scale,unit,data_origin
0,2025-01-01,Aido Rover,AR-01,incident_detection_pct,75.000000,91.000000,100,%,SYNTHETIC
1,2025-01-01,Aido Rover,AR-01,false_alert_pct,2.000000,77.000000,100,%,SYNTHETIC
2,2025-01-01,Aido Rover,AR-01,patrol_coverage_pct,460.182939,493.867475,100,%,SYNTHETIC
3,2025-01-01,Aido Rover,AR-01,uptime_pct,357.818275,372.000000,100,%,SYNTHETIC
4,2025-01-01,Aido Rover,AR-01,energy_kwh_per_hour,70.066812,314.175000,1,kWh/hour,SYNTHETIC


In [6]:
measurement_keys = [
    "platform", "unit_id", "month", "metric"
]

clean_measurements = raw_measurements.copy()

rows_before = len(clean_measurements)

clean_measurements = (
    clean_measurements.drop_duplicates().copy()
)

duplicates_removed = (
    rows_before - len(clean_measurements)
)


clean_measurements["month"] = pd.to_datetime(
    clean_measurements["month"],
    format="%Y-%m-%d",
    errors="raise"
)

assert clean_measurements[
    measurement_keys
].notna().all().all(), "Missing measurement identifier."

assert not clean_measurements.duplicated(
    measurement_keys
).any(), "Conflicting duplicate measurement keys."


for column in ["numerator", "denominator", "scale"]:

    clean_measurements[column] = pd.to_numeric(
        clean_measurements[column],
        errors="raise"
    )

    observed = clean_measurements[column].dropna()

    assert np.isfinite(observed).all(), (
        f"Non-finite values found in {column}."
    )

    assert observed.ge(0).all(), (
        f"Negative values found in {column}."
    )


assert clean_measurements["scale"].notna().all()
assert clean_measurements["scale"].gt(0).all()


valid_input = (
    clean_measurements["numerator"].notna()
    & clean_measurements["denominator"].gt(0)
)

clean_measurements["value"] = np.nan

clean_measurements.loc[valid_input, "value"] = (
    clean_measurements.loc[valid_input, "numerator"]
    / clean_measurements.loc[valid_input, "denominator"]
    * clean_measurements.loc[valid_input, "scale"]
)

clean_measurements["quality_flag"] = np.where(
    valid_input,
    "OK",
    "MISSING_INPUT_OR_ZERO_DENOMINATOR"
)


percentage_rows = (
    clean_measurements["unit"].eq("%")
    & valid_input
)

assert clean_measurements.loc[
    percentage_rows, "value"
].between(0, 100).all(), "Percentage outside 0–100."


print(f"Exact duplicates removed: {duplicates_removed}")

display(
    clean_measurements["quality_flag"]
    .value_counts()
    .rename_axis("quality_flag")
    .to_frame("records")
)

Exact duplicates removed: 0


,records
quality_flag,
OK,900


In [7]:
clean_measurements = (
    clean_measurements
    .sort_values(
        ["platform", "unit_id", "metric", "month"]
    )
    .reset_index(drop=True)
)


group_keys = ["platform", "unit_id", "metric"]

metric_groups = clean_measurements.groupby(
    group_keys
)["value"]


clean_measurements["rolling_4_month_mean"] = (
    metric_groups.transform(
        lambda series: series.rolling(
            4, min_periods=4
        ).mean()
    )
)

three_month_lag = metric_groups.shift(3)

clean_measurements["change_vs_3_months_ago"] = (
    clean_measurements["value"] - three_month_lag
)


clean_measurements["change_unit"] = np.where(
    clean_measurements["unit"].eq("%"),
    "percentage points",
    clean_measurements["unit"]
)


display(
    clean_measurements[
        [
            "month",
            "platform",
            "unit_id",
            "metric",
            "value",
            "rolling_4_month_mean",
            "change_vs_3_months_ago",
            "change_unit"
        ]
    ].head(12)
)

,month,platform,unit_id,metric,value,rolling_4_month_mean,change_vs_3_months_ago,change_unit
0,2025-01-01,Aido Rover,AR-01,energy_kwh_per_hour,0.223018,NaN,NaN,kWh/hour
1,2025-02-01,Aido Rover,AR-01,energy_kwh_per_hour,0.176724,NaN,NaN,kWh/hour
2,2025-03-01,Aido Rover,AR-01,energy_kwh_per_hour,0.172132,NaN,NaN,kWh/hour
3,2025-04-01,Aido Rover,AR-01,energy_kwh_per_hour,0.232042,0.200979,0.009024,kWh/hour
4,2025-05-01,Aido Rover,AR-01,energy_kwh_per_hour,0.245908,0.206702,0.069184,kWh/hour
5,2025-06-01,Aido Rover,AR-01,energy_kwh_per_hour,0.210438,0.215130,0.038306,kWh/hour
6,2025-07-01,Aido Rover,AR-01,energy_kwh_per_hour,0.256544,0.236233,0.024502,kWh/hour
7,2025-08-01,Aido Rover,AR-01,energy_kwh_per_hour,0.123634,0.209131,-0.122274,kWh/hour
8,2025-09-01,Aido Rover,AR-01,energy_kwh_per_hour,0.168242,0.189715,-0.042196,kWh/hour
9,2025-10-01,Aido Rover,AR-01,energy_kwh_per_hour,0.191898,0.185080,-0.064647,kWh/hour


## Feature Interpretation

The four-month rolling feature is an arithmetic mean of monthly KPI values.
It describes the typical monthly result and is not a pooled four-month rate.

For exposure-weighted rates, the appropriate numerator and denominator
should be aggregated before division. Sentinel MTBF-based availability
requires separate reliability aggregation and should not be pooled as if
MTBF were a monthly exposure total.

The three-month comparison is a change from the observation three months
earlier, not a comparison of aggregated calendar quarters. Percentage metrics
use percentage-point changes; other metrics retain their original units.

The first three months have no rolling-four-month or three-month-lag value.
These missing values are expected and are not imputed.

In [8]:
record_keys = ["month", "platform", "unit_id"]


simulated_dataset = (
    clean_measurements
    .pivot(
        index=record_keys,
        columns="metric",
        values="value"
    )
    .reset_index()
)

simulated_dataset.columns.name = None

simulated_dataset["data_origin"] = "SYNTHETIC"

simulated_dataset = (
    simulated_dataset
    .sort_values(record_keys)
    .reset_index(drop=True)
)


# Attach framework definitions, assumptions, and source references.
measurement_audit = clean_measurements.merge(
    metadata[
        [
            "platform",
            "metric",
            "kpi_name",
            "numerator",
            "denominator",
            "public_proxy_benchmark",
            "source",
            "limitation",
            "simulation_assumption",
            "implementation_note"
        ]
    ].rename(
        columns={
            "numerator": "framework_numerator_definition",
            "denominator": "framework_denominator_definition"
        }
    ),
    on=["platform", "metric"],
    how="left",
    validate="many_to_one"
)


display(simulated_dataset.head())

print("Main dataset shape:", simulated_dataset.shape)

display(
    simulated_dataset.groupby("platform")
    .size()
    .rename("records")
    .to_frame()
)

,month,platform,unit_id,active_day_engagement_pct,availability_pct,energy_kwh_per_hour,false_alert_pct,goal_success_pct,incident_detection_pct,interactions_per_active_user_day,monitoring_coverage_pct,operator_response_min,patrol_coverage_pct,response_latency_sec,session_duration_min,uptime_pct,data_origin
0,2025-01-01,Aido Rover,AR-01,NaN,NaN,0.223018,2.597403,NaN,82.417582,NaN,NaN,NaN,93.179438,NaN,NaN,96.187708,SYNTHETIC
1,2025-01-01,Aido Rover,AR-02,NaN,NaN,0.197169,1.724138,NaN,85.074627,NaN,NaN,NaN,94.795528,NaN,NaN,94.134543,SYNTHETIC
2,2025-01-01,Aido Rover,AR-03,NaN,NaN,0.128541,0.000000,NaN,93.684211,NaN,NaN,NaN,88.243202,NaN,NaN,94.326833,SYNTHETIC
3,2025-01-01,Aido Rover,AR-04,NaN,NaN,0.224020,3.896104,NaN,98.666667,NaN,NaN,NaN,89.223825,NaN,NaN,99.894718,SYNTHETIC
4,2025-01-01,Aido Rover,AR-05,NaN,NaN,0.240853,5.405405,NaN,86.419753,NaN,NaN,NaN,93.154825,NaN,NaN,96.316179,SYNTHETIC


Main dataset shape: (180, 17)


,records
platform,
Aido Rover,60
Fari,60
Sentinel Prime AI,60


In [9]:
validation_checks = {
    "180 monthly deployment records":
        len(simulated_dataset) == 180,

    "900 KPI measurement records":
        len(clean_measurements) == 900,

    "60 monthly records per platform":
        simulated_dataset.groupby("platform").size().eq(60).all(),

    "12 months per deployment unit":
        simulated_dataset.groupby(
            ["platform", "unit_id"]
        )["month"].nunique().eq(12).all(),

    "Five KPI measurements per deployment-month":
        clean_measurements.groupby(
            record_keys
        ).size().eq(5).all(),

    "All 15 platform-KPI combinations represented":
        clean_measurements[
            ["platform", "metric"]
        ].drop_duplicates().shape[0] == 15,

    "No duplicate monthly records":
        not simulated_dataset.duplicated(record_keys).any(),

    "All operational records labeled synthetic":
        simulated_dataset["data_origin"].eq("SYNTHETIC").all(),

    "Source references retained":
        measurement_audit["source"].notna().all(),

    "Simulation assumptions retained":
        measurement_audit["simulation_assumption"].notna().all()
}


validation_table = pd.DataFrame({
    "check": list(validation_checks),
    "status": [
        "PASS" if passed else "FAIL"
        for passed in validation_checks.values()
    ]
})

display(validation_table)

assert all(validation_checks.values()), (
    "Validation failed. Review the failed checks before export."
)


output_tables = {
    "W03_Simulated_Dataset.csv": simulated_dataset,
    "W03_KPI_Measurement_Audit.csv": measurement_audit,
    "W03_KPI_Metadata.csv": metadata,
    "W03_Public_Financial_Features.csv": public_financials
}

for filename, table in output_tables.items():

    output_path = OUTPUT_DIR / filename

    table.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d"
    )

    print(f"Saved: {output_path}")


elapsed_seconds = perf_counter() - pipeline_start

print(f"\nElapsed notebook time: {elapsed_seconds:.2f} seconds")
print("Pipeline completed successfully.")

,check,status
0,180 monthly deployment records,PASS
1,900 KPI measurement records,PASS
2,60 monthly records per platform,PASS
3,12 months per deployment unit,PASS
4,Five KPI measurements per deployment-month,PASS
5,All 15 platform-KPI combinations represented,PASS
6,No duplicate monthly records,PASS
7,All operational records labeled synthetic,PASS
8,Source references retained,PASS
9,Simulation assumptions retained,PASS


Saved: outputs\W03_Simulated_Dataset.csv
Saved: outputs\W03_KPI_Measurement_Audit.csv
Saved: outputs\W03_KPI_Metadata.csv
Saved: outputs\W03_Public_Financial_Features.csv

Elapsed notebook time: 0.66 seconds
Pipeline completed successfully.


## Output Summary

The pipeline produces:

- `W03_Simulated_Dataset.csv`: 180 monthly records for the dashboard prototype.
- `W03_KPI_Measurement_Audit.csv`: 900 KPI measurements with calculation inputs,
  rolling features, quality flags, assumptions, and source references.
- `W03_KPI_Metadata.csv`: metric definitions, implementation decisions,
  simulation assumptions, and public proxy references.
- `W03_Public_Financial_Features.csv`: public financial context with quarterly
  revenue growth and a four-quarter rolling revenue average.

## Limitations and Next Steps

All operational observations are synthetic. Public research benchmarks,
commercial specifications, and service-level targets differ in their
definitions and operating environments. They provide contextual references,
not validated estimates of InGen performance.

The prototype does not demonstrate actual seasonality, operational improvement,
customer outcomes, or causal relationships. Incident definitions, alert-review
rules, qualifying interaction criteria, and exposure denominators would need
to be confirmed before using the pipeline with production data.

The next step is to use the synthetic dataset to build and test the Week 4
dashboard, while retaining clear synthetic-data labels and platform-specific
metric definitions.